<a href="https://colab.research.google.com/github/Lena24668/Ant_Tracking/blob/main/Preannotation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# 1) SETUP
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =========================================================
# 2) PAKETE INSTALLIEREN
# =========================================================

!pip install -q ultralytics pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.4 MB/s eta 0:00:00


In [ ]:
# =========================================================
# 3) PARAMETER
#    NUR DIESE WERTE SPAETER ANPASSEN
# =========================================================

from pathlib import Path

# Pfad zu deinem Modell auf Drive
MODEL_PATH = "/content/drive/MyDrive/models/HangarAnt_yolo26s_pose.pt"

# Name der Klasse in Roboflow
CLASS_NAME = "Ants"

# Reihenfolge deiner Keypoints
# HIER ANPASSEN
KEYPOINT_NAMES = [
    "head",
    "tail"
]

# Confidence fuer Objekt-Erkennung
PREDICT_CONF = 0.10

# IoU fuer die eingebaute YOLO-NMS
PREDICT_IOU = 0.0

# Eigener Duplicate-Filter nach der Prediction
FILTER_IOU = 0.0
CLASS_AWARE_FILTER = True

# Confidence-Schwelle fuer einzelne Keypoints:
# Wenn kp_conf >= KPT_CONF_TO_VISIBLE -> visibility = 2
# Sonst -> visibility = 0
KPT_CONF_TO_VISIBLE = 0.05

# Bildgroesse fuer Prediction
IMGSZ = 640

# Arbeitsordner
WORKDIR = Path("/content/project")
RAW_ZIP_DIR = WORKDIR / "uploaded_zip"
EXTRACT_DIR = WORKDIR / "extracted"
PREDICT_DIR = WORKDIR / "predict_output"
FINAL_DATASET_DIR = WORKDIR / "roboflow_dataset"

for p in [RAW_ZIP_DIR, EXTRACT_DIR, PREDICT_DIR, FINAL_DATASET_DIR]:
    p.mkdir(parents=True, exist_ok=True)

NUM_KEYPOINTS = len(KEYPOINT_NAMES)

print("MODEL_PATH:", MODEL_PATH)
print("CLASS_NAME:", CLASS_NAME)
print("KEYPOINT_NAMES:", KEYPOINT_NAMES)
print("NUM_KEYPOINTS:", NUM_KEYPOINTS)
print("PREDICT_CONF:", PREDICT_CONF)
print("PREDICT_IOU:", PREDICT_IOU)
print("FILTER_IOU:", FILTER_IOU)
print("CLASS_AWARE_FILTER:", CLASS_AWARE_FILTER)
print("KPT_CONF_TO_VISIBLE:", KPT_CONF_TO_VISIBLE)
print("IMGSZ:", IMGSZ)

MODEL_PATH: /content/drive/MyDrive/models/HangarAnt_yolo26s_pose.pt
CLASS_NAME: Ants
KEYPOINT_NAMES: ['head', 'tail']
NUM_KEYPOINTS: 2
PREDICT_CONF: 0.1
PREDICT_IOU: 0.0
FILTER_IOU: 0.0
CLASS_AWARE_FILTER: True
KPT_CONF_TO_VISIBLE: 0.05
IMGSZ: 640


In [ ]:
# =========================================================
# 4) ZIP AUS GOOGLE DRIVE VERWENDEN
# =========================================================

from pathlib import Path

# Pfad zu deiner ZIP auf Google Drive
ZIP_PATH = "/content/drive/MyDrive/Ant_Stuff/batch_0001.zip"

zip_path = Path(ZIP_PATH)

if not zip_path.exists():
    raise FileNotFoundError(f"ZIP nicht gefunden: {zip_path}")

print("ZIP gefunden:", zip_path)

ZIP gefunden: /content/drive/MyDrive/Ant_Stuff/batch_0001.zip


In [ ]:
# =========================================================
# 5) ZIP ENT PACKEN
#    SUCHT AUTOMATISCH DIE HOCHGELADENE ZIP
#    ENTFERNT MAC-METADATEN
# =========================================================

import zipfile
import shutil
from pathlib import Path

zip_files = [zip_path]

if len(zip_files) == 0:
    raise FileNotFoundError("Keine ZIP-Datei in /content gefunden.")

if len(zip_files) > 1:
    print("Mehrere ZIP-Dateien gefunden. Es wird die neueste verwendet.")

zip_path = max(zip_files, key=lambda p: p.stat().st_mtime)
print("Verwendete ZIP:", zip_path)

# alten Extract-Ordner komplett löschen
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# ZIP entpacken
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(EXTRACT_DIR)

print("Entpackt nach:", EXTRACT_DIR)

# ---------------------------------------------------------
# Mac-Metadaten entfernen (._ Dateien und __MACOSX Ordner)
# ---------------------------------------------------------

removed_files = 0

for p in EXTRACT_DIR.rglob("._*"):
    if p.is_file():
        p.unlink()
        removed_files += 1

macosx_dir = EXTRACT_DIR / "__MACOSX"
if macosx_dir.exists():
    shutil.rmtree(macosx_dir)

print("Mac-Metadaten entfernt:", removed_files, "Dateien")
print("Ordner __MACOSX entfernt (falls vorhanden)")

Verwendete ZIP: /content/drive/MyDrive/Ant_Stuff/batch_0001.zip
Entpackt nach: /content/project/extracted
Mac-Metadaten entfernt: 3001 Dateien
Ordner __MACOSX entfernt (falls vorhanden)


In [ ]:
# =========================================================
# 6) BILDORDNER AUTOMATISCH FINDEN
# =========================================================

from pathlib import Path
import os

image_exts = {".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"}

candidate_dirs = []
for root, dirs, files_ in os.walk(EXTRACT_DIR):
    n_images = sum(1 for f in files_ if Path(f).suffix in image_exts)
    if n_images > 0:
        candidate_dirs.append((Path(root), n_images))

if not candidate_dirs:
    raise FileNotFoundError("Kein Ordner mit Bildern gefunden.")

candidate_dirs = sorted(candidate_dirs, key=lambda x: x[1], reverse=True)

print("Gefundene Ordner mit Bildern:")
for folder, n in candidate_dirs[:20]:
    print(f"{folder} -> {n} Bilder")

IMAGE_DIR = candidate_dirs[0][0]

print("\nVerwendeter Bildordner:", IMAGE_DIR)
print("Anzahl Bilder dort:", candidate_dirs[0][1])

image_files = sorted(
    [p for p in IMAGE_DIR.iterdir() if p.is_file() and p.suffix in image_exts]
)

print("Gesamtzahl Bilder:", len(image_files))
print("Erste 10 Bilder:")
for p in image_files[:10]:
    print(p.name)

Gefundene Ordner mit Bildern:
/content/project/extracted/batch_0001 -> 1500 Bilder

Verwendeter Bildordner: /content/project/extracted/batch_0001
Anzahl Bilder dort: 1500
Gesamtzahl Bilder: 1500
Erste 10 Bilder:
tile_000000_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f860_x5390_y5226.png
tile_000001_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f860_x5191_y3772.png
tile_000002_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f860_x3092_y5734.png
tile_000003_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f860_x466_y5334.png
tile_000004_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f860_x4426_y5578.png
tile_000005_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f3444_x3171_y2919.png
tile_000006_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f3444_x130_y1685.png
tile_000007_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f3444_x769_y2391.png
tile_000008_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f3444_x5611_y2433.png
tile_000009_VTKA_AuPa_20251127-185459-699747-1542

In [ ]:
# =========================================================
# 7) HILFSFUNKTIONEN FUER DUPLIKAT-FILTER
# =========================================================

import torch

def box_iou_xyxy(box1, box2):
    """
    box1: Tensor (4,)
    box2: Tensor (N, 4)
    returns: IoU Tensor (N,)
    """
    x1 = torch.maximum(box1[0], box2[:, 0])
    y1 = torch.maximum(box1[1], box2[:, 1])
    x2 = torch.minimum(box1[2], box2[:, 2])
    y2 = torch.minimum(box1[3], box2[:, 3])

    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area1 = (box1[2] - box1[0]).clamp(min=0) * (box1[3] - box1[1]).clamp(min=0)
    area2 = (box2[:, 2] - box2[:, 0]).clamp(min=0) * (box2[:, 3] - box2[:, 1]).clamp(min=0)
    union = area1 + area2 - inter + 1e-9
    return inter / union

def filter_duplicate_boxes(boxes, iou_thr=0.2, class_aware=True):
    """
    Eigener Duplikat-Filter nach der YOLO-Prediction.
    Behaelt jeweils die Box mit der hoeheren Confidence.
    """
    if boxes is None or len(boxes) == 0:
        return []

    xyxy = boxes.xyxy.detach().cpu()
    conf = boxes.conf.detach().cpu()
    cls = boxes.cls.detach().cpu()

    order = torch.argsort(conf, descending=True)
    keep = []

    while len(order) > 0:
        i = int(order[0].item())
        keep.append(i)

        if len(order) == 1:
            break

        rest = order[1:]
        ious = box_iou_xyxy(xyxy[i], xyxy[rest])

        if class_aware:
            same_class = cls[rest] == cls[i]
            suppress = (ious > iou_thr) & same_class
        else:
            suppress = ious > iou_thr

        order = rest[~suppress]

    return keep

In [ ]:
image_files = []
for ext in image_exts:
    image_files.extend(IMAGE_DIR.glob(f"*{ext}"))

image_files = sorted(image_files)
print("Gesamtzahl Bilder:", len(image_files))

Gesamtzahl Bilder: 1500


In [ ]:
# =========================================================
# 8) YOLO-POSE-PREDICTION MIT EIGENEM DUPLIKAT-FILTER
# =========================================================

from ultralytics import YOLO
from pathlib import Path

YOLO_LABEL_DIR = PREDICT_DIR / "run" / "labels"
YOLO_LABEL_DIR.mkdir(parents=True, exist_ok=True)

PREVIEW_DIR = PREDICT_DIR / "run" / "preview"
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO(MODEL_PATH)

results = model.predict(
    source=str(IMAGE_DIR),
    conf=PREDICT_CONF,
    iou=PREDICT_IOU,
    imgsz=IMGSZ,
    save=False,
    verbose=False
)

print("Anzahl Prediction-Resultate:", len(results))

total_raw = 0
total_kept = 0

for res in results:
    img_path = Path(res.path)
    out_txt = YOLO_LABEL_DIR / f"{img_path.stem}.txt"

    boxes = res.boxes
    keep = filter_duplicate_boxes(
        boxes=boxes,
        iou_thr=FILTER_IOU,
        class_aware=CLASS_AWARE_FILTER
    )

    n_raw = 0 if boxes is None else len(boxes)
    n_kept = len(keep)

    total_raw += n_raw
    total_kept += n_kept

    with open(out_txt, "w") as f:
        if boxes is not None and len(keep) > 0:
            xywhn = boxes.xywhn.detach().cpu()
            cls = boxes.cls.detach().cpu()

            kpts_xyn = None
            kpts_conf = None

            if getattr(res, "keypoints", None) is not None:
                if getattr(res.keypoints, "xyn", None) is not None:
                    kpts_xyn = res.keypoints.xyn.detach().cpu()   # normalisierte x,y
                if getattr(res.keypoints, "conf", None) is not None:
                    kpts_conf = res.keypoints.conf.detach().cpu() # keypoint-confidence

            for idx in keep:
                cls_id = int(cls[idx].item())
                x, y, w, h = xywhn[idx].tolist()

                line = [str(cls_id), f"{x:.6f}", f"{y:.6f}", f"{w:.6f}", f"{h:.6f}"]

                if kpts_xyn is not None:
                    for kp_i in range(kpts_xyn.shape[1]):
                        kp_x = float(kpts_xyn[idx, kp_i, 0].item())
                        kp_y = float(kpts_xyn[idx, kp_i, 1].item())

                        if kpts_conf is not None:
                            kp_conf = float(kpts_conf[idx, kp_i].item())
                        else:
                            kp_conf = 1.0

                        line.extend([f"{kp_x:.6f}", f"{kp_y:.6f}", f"{kp_conf:.6f}"])

                f.write(" ".join(line) + "\n")

    print(f"{img_path.name}: raw={n_raw}, kept={n_kept}, removed={n_raw - n_kept}")

print("\nGesamt:")
print("Rohe Detections:", total_raw)
print("Nach Duplicate-Filter behalten:", total_kept)
print("Als Duplikate entfernt:", total_raw - total_kept)
print("Gefilterte Label-Dateien gespeichert in:", YOLO_LABEL_DIR)

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



In [ ]:
test_labels = sorted(YOLO_LABEL_DIR.glob("*.txt"))
print("Anzahl Label-Dateien:", len(test_labels))

if test_labels:
    print("\nBeispiel aus:", test_labels[0].name)
    with open(test_labels[0], "r") as f:
        for i, line in enumerate(f):
            print(line.strip())
            if i >= 2:
                break

Anzahl Label-Dateien: 3000

Beispiel aus: tile_000000_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x5147_y5739.txt


In [ ]:
# =========================================================
# 9) PREDICTION-LABELS FINDEN
# =========================================================

from pathlib import Path

if not YOLO_LABEL_DIR.exists():
    raise FileNotFoundError(f"Label-Ordner nicht gefunden: {YOLO_LABEL_DIR}")

label_files = sorted(YOLO_LABEL_DIR.glob("*.txt"))
print("Anzahl YOLO-Labeldateien:", len(label_files))
print("Erste 10 Labeldateien:")
for p in label_files[:10]:
    print(p.name)

Anzahl YOLO-Labeldateien: 3000
Erste 10 Labeldateien:
tile_000000_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x5147_y5739.txt
tile_000000_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f860_x5390_y5226.txt
tile_000001_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x3627_y1363.txt
tile_000001_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f860_x5191_y3772.txt
tile_000002_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x1981_y5450.txt
tile_000002_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f860_x3092_y5734.txt
tile_000003_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x1663_y1529.txt
tile_000003_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f860_x466_y5334.txt
tile_000004_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x2038_y5592.txt
tile_000004_VTKA_AuPa_20251127-185459-699747-1542228-1548227_f860_x4426_y5578.txt


In [ ]:
# =========================================================
# 10) YOLO-PREDICTIONS IN ROBOFLOW-TAUGLICHES FORMAT UMWANDELN
#     AUS: class box_x box_y box_w box_h kp_x kp_y kp_conf ...
#     WIRD: class box_x box_y box_w box_h kp_x kp_y visibility ...
# =========================================================

from pathlib import Path

CONVERTED_LABEL_DIR = WORKDIR / "converted_labels"
CONVERTED_LABEL_DIR.mkdir(parents=True, exist_ok=True)

converted_count = 0
skipped_lines = 0

for txt_file in YOLO_LABEL_DIR.glob("*.txt"):
    new_lines = []

    with open(txt_file, "r") as f:
        for line in f:
            vals = line.strip().split()
            if not vals:
                continue

            cls_id = vals[0]
            box = vals[1:5]
            rest = vals[5:]

            # Es muss ein Mehrfaches von 3 sein: x y conf
            if len(rest) % 3 != 0:
                skipped_lines += 1
                continue

            n_kpts_in_file = len(rest) // 3
            if n_kpts_in_file != NUM_KEYPOINTS:
                print(f"Warnung in {txt_file.name}: Datei hat {n_kpts_in_file} Keypoints, erwartet sind {NUM_KEYPOINTS}.")
                skipped_lines += 1
                continue

            converted = [cls_id] + box

            for i in range(0, len(rest), 3):
                x = rest[i]
                y = rest[i + 1]
                conf = float(rest[i + 2])

                vis = "2" if conf >= KPT_CONF_TO_VISIBLE else "0"
                converted.extend([x, y, vis])

            new_lines.append(" ".join(converted))

    with open(CONVERTED_LABEL_DIR / txt_file.name, "w") as f:
        for line in new_lines:
            f.write(line + "\n")

    converted_count += 1

print("Konvertierte Label-Dateien:", converted_count)
print("Uebersprungene Zeilen:", skipped_lines)
print("Konvertierte Labels liegen hier:", CONVERTED_LABEL_DIR)

Konvertierte Label-Dateien: 3000
Uebersprungene Zeilen: 0
Konvertierte Labels liegen hier: /content/project/converted_labels


In [ ]:
# =========================================================
# 11) KONTROLL-CHECK EINER KONVERTIERTEN DATEI
# =========================================================

converted_files = sorted(CONVERTED_LABEL_DIR.glob("*.txt"))
if converted_files:
    test_file = converted_files[0]
    print("Beispiel-Datei:", test_file.name)
    with open(test_file, "r") as f:
        for i, line in enumerate(f):
            print(line.strip())
            if i >= 2:
                break
else:
    print("Keine konvertierten Labels gefunden.")

Beispiel-Datei: tile_000000_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x5147_y5739.txt


In [ ]:
# =========================================================
# 12) FINALES ROBOFLOW-DATASET ERZEUGEN
#     STRUKTUR:
#     roboflow_dataset/
#       images/
#       labels/
#       data.yaml
# =========================================================

import shutil
from pathlib import Path

# alten Inhalt loeschen
if FINAL_DATASET_DIR.exists():
    shutil.rmtree(FINAL_DATASET_DIR)

(FINAL_DATASET_DIR / "images").mkdir(parents=True, exist_ok=True)
(FINAL_DATASET_DIR / "labels").mkdir(parents=True, exist_ok=True)

# Bilder kopieren
for img in image_files:
    shutil.copy2(img, FINAL_DATASET_DIR / "images" / img.name)

# Labels kopieren
for lab in CONVERTED_LABEL_DIR.glob("*.txt"):
    shutil.copy2(lab, FINAL_DATASET_DIR / "labels" / lab.name)

print("Finales Dataset erstellt in:", FINAL_DATASET_DIR)

Finales Dataset erstellt in: /content/project/roboflow_dataset


In [ ]:
# =========================================================
# 13) DATA.YAML SCHREIBEN
#     HILFT FUER DOKUMENTATION UND WEITERE VERWENDUNG
# =========================================================

import yaml

data_yaml = {
    "path": str(FINAL_DATASET_DIR),
    "train": "images",
    "val": "images",
    "names": {
        0: CLASS_NAME
    },
    "kpt_shape": [NUM_KEYPOINTS, 3],
    "keypoints": {i: name for i, name in enumerate(KEYPOINT_NAMES)},
}

with open(FINAL_DATASET_DIR / "data.yaml", "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print("data.yaml geschrieben nach:", FINAL_DATASET_DIR / "data.yaml")

data.yaml geschrieben nach: /content/project/roboflow_dataset/data.yaml


In [ ]:
# =========================================================
# 14) VALIDIERUNG
#     PRUEFT:
#     - WIE VIELE BILDER GIBT ES?
#     - WIE VIELE LABELS GIBT ES?
#     - HABEN LABELS PASSENDE BILDER?
# =========================================================

from pathlib import Path

final_images = sorted((FINAL_DATASET_DIR / "images").glob("*"))
final_labels = sorted((FINAL_DATASET_DIR / "labels").glob("*.txt"))

image_stems = {p.stem for p in final_images if p.suffix in image_exts}
label_stems = {p.stem for p in final_labels}

matched = image_stems & label_stems
images_without_labels = image_stems - label_stems
labels_without_images = label_stems - image_stems

print("Anzahl Bilder:", len(image_stems))
print("Anzahl Labels:", len(label_stems))
print("Paare Bild+Label:", len(matched))
print("Bilder ohne Label:", len(images_without_labels))
print("Labels ohne Bild:", len(labels_without_images))

if images_without_labels:
    print("\nErste 10 Bilder ohne Label:")
    for x in sorted(list(images_without_labels))[:10]:
        print(x)

if labels_without_images:
    print("\nErste 10 Labels ohne Bild:")
    for x in sorted(list(labels_without_images))[:10]:
        print(x)

Anzahl Bilder: 1500
Anzahl Labels: 3000
Paare Bild+Label: 1500
Bilder ohne Label: 0
Labels ohne Bild: 1500

Erste 10 Labels ohne Bild:
tile_000000_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x5147_y5739
tile_000001_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x3627_y1363
tile_000002_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x1981_y5450
tile_000003_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x1663_y1529
tile_000004_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f4891_x2038_y5592
tile_000005_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f3302_x2237_y5504
tile_000006_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f3302_x5496_y1306
tile_000007_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f3302_x4029_y2675
tile_000008_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f3302_x5068_y5864
tile_000009_VTKA_AuPa_20251127-131459-956311-1134194-1140193_f3302_x1282_y709


In [ ]:
# =========================================================
# 15) DOWNLOAD-PAKET BAUEN
#     DAS IST NUR FUER DEN DOWNLOAD AUS COLAB
# =========================================================

import shutil

archive_base = "/content/roboflow_dataset"
archive_path = shutil.make_archive(archive_base, "zip", root_dir=FINAL_DATASET_DIR)

print("ZIP erstellt:", archive_path)

ZIP erstellt: /content/roboflow_dataset.zip


In [ ]:
# =========================================================
# 16) ZIP HERUNTERLADEN
#     AUF DEM MAC ENTPACKEN
#     DANN IN ROBOFLOW DIE ORDNER images/ und labels/ HOCHLADEN
# =========================================================

from google.colab import files
files.download("/content/roboflow_dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>